##Imports


In [1]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

##Dataset


In [2]:
from google.colab import drive
drive.mount('/content/drive')

data = pd.read_csv("/content/drive/Shareddrives/CS133 Project/dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")

Mounted at /content/drive


##Preprocessing


In [3]:
# Strip white space in column names
data.columns = data.columns.str.strip()

In [4]:
# Find null values (Only 4 entries have null values in Flow Bytes/s)
null_counts = data.isnull().sum()
null_counts[null_counts > 0]

,0
Flow Bytes/s,4


In [5]:
# Remove entries with null values
data.dropna(inplace=True, axis='rows')

In [6]:
# Drop Flow ID and Timestamp (not helpful in detecting DDOS)
if 'Flow ID' in data.columns:
    data.drop('Flow ID', axis=1, inplace=True)
if 'Timestamp' in data.columns:
    data.drop('Timestamp', axis=1, inplace=True)

# Drop columns that only have only one value
cols_to_drop = data.columns[data.nunique() == 1]
data.drop(cols_to_drop, axis=1, inplace=True)

In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 225741 entries, 0 to 225744
Data columns (total 73 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Source IP                    225741 non-null  object 
 1   Source Port                  225741 non-null  int64  
 2   Destination IP               225741 non-null  object 
 3   Destination Port             225741 non-null  int64  
 4   Protocol                     225741 non-null  int64  
 5   Flow Duration                225741 non-null  int64  
 6   Total Fwd Packets            225741 non-null  int64  
 7   Total Backward Packets       225741 non-null  int64  
 8   Total Length of Fwd Packets  225741 non-null  int64  
 9   Total Length of Bwd Packets  225741 non-null  int64  
 10  Fwd Packet Length Max        225741 non-null  int64  
 11  Fwd Packet Length Min        225741 non-null  int64  
 12  Fwd Packet Length Mean       225741 non-null  float64
 13  Fwd 

## Machine Learning

In [8]:
target = data['Label']
features = data.drop('Label', axis=1)

In [9]:
le = LabelEncoder()

# Apply LabelEncoder to IP address columns
features['Source IP'] = le.fit_transform(features['Source IP'])
features['Destination IP'] = le.fit_transform(features['Destination IP'])

# Display the encoded columns
display(features[['Source IP', 'Destination IP']].head())

# Replace infinity values with the maximum finite value for each column
features.replace([np.inf, -np.inf], np.nan, inplace=True)
features.fillna(features.max(), inplace=True)

,Source IP,Destination IP
0,22,866
1,37,866
2,37,866
3,45,860
4,49,866


# Machine Learning: DecisionTree Classifier

In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics

# Method 1: StratifiedShuffleSplit
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_dex, test_dex in split.split(features, target):
    X_train, X_test = features.iloc[train_dex], features.iloc[test_dex]
    y_train, y_test = target.iloc[train_dex], target.iloc[test_dex]

clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("=== Decision Tree Metrics (StratifiedShuffleSplit) ===")
print("Accuracy:", metrics.accuracy_score(y_test, y_pred))
print("Precision:", metrics.precision_score(y_test, y_pred, average='weighted'))
print("Recall:", metrics.recall_score(y_test, y_pred, average='weighted'))
print(f"F1 Score:", metrics.f1_score(y_test, y_pred, average='weighted'))

print("\nConfusion Matrix:")
print(metrics.confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(metrics.classification_report(y_test, y_pred))

=== Decision Tree Metrics (StratifiedShuffleSplit) ===
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

Confusion Matrix:
[[19543     0]
 [    0 25606]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     19543
        DDoS       1.00      1.00      1.00     25606

    accuracy                           1.00     45149
   macro avg       1.00      1.00      1.00     45149
weighted avg       1.00      1.00      1.00     45149



In [12]:
# Method 2: train_test_split
from sklearn.tree import DecisionTreeClassifier

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42
)
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("\n\n=== Decision Tree Metrics (train_test_split) ===")
print("Accuracy:", metrics.accuracy_score(y_test, y_pred))
print("Precision:", metrics.precision_score(y_test, y_pred, average='weighted'))
print("Recall:", metrics.recall_score(y_test, y_pred, average='weighted'))
print(f"F1 Score:", metrics.f1_score(y_test, y_pred, average='weighted'))

print("\nConfusion Matrix:")
print(metrics.confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(metrics.classification_report(y_test, y_pred))



=== Decision Tree Metrics (train_test_split) ===
Accuracy: 0.9999557022303927
Precision: 0.9999557067927048
Recall: 0.9999557022303927
F1 Score: 0.9999557025102433

Confusion Matrix:
[[19417     0]
 [    2 25730]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     19417
        DDoS       1.00      1.00      1.00     25732

    accuracy                           1.00     45149
   macro avg       1.00      1.00      1.00     45149
weighted avg       1.00      1.00      1.00     45149



In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("=== Random Forest Metrics ===")
print("Accuracy:", metrics.accuracy_score(y_test, y_pred))
print("Precision:", metrics.precision_score(y_test, y_pred, average='weighted'))
print("Recall:", metrics.recall_score(y_test, y_pred, average='weighted'))
print("F1 Score:", metrics.f1_score(y_test, y_pred, average='weighted'))

print("\nConfusion Matrix:")
print(metrics.confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(metrics.classification_report(y_test, y_pred))

=== Random Forest Metrics ===
Accuracy: 0.9999557022303927
Precision: 0.9999557067927048
Recall: 0.9999557022303927
F1 Score: 0.9999557025102433

Confusion Matrix:
[[19417     0]
 [    2 25730]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     19417
        DDoS       1.00      1.00      1.00     25732

    accuracy                           1.00     45149
   macro avg       1.00      1.00      1.00     45149
weighted avg       1.00      1.00      1.00     45149



In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print("\n\n=== Logistic Regression Metrics ===")
print("Accuracy:", metrics.accuracy_score(y_test, y_pred))
print("Precision:", metrics.precision_score(y_test, y_pred, average='weighted'))
print("Recall:", metrics.recall_score(y_test, y_pred, average='weighted'))
print(f"F1 Score:", metrics.f1_score(y_test, y_pred, average='weighted'))

print("\nConfusion Matrix:")
print(metrics.confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(metrics.classification_report(y_test, y_pred))



=== Logistic Regression Metrics ===
Accuracy: 0.9994462778799087
Precision: 0.9994463023682426
Recall: 0.9994462778799087
F1 Score: 0.9994462656182855

Confusion Matrix:
[[19401    16]
 [    9 25723]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     19417
        DDoS       1.00      1.00      1.00     25732

    accuracy                           1.00     45149
   macro avg       1.00      1.00      1.00     45149
weighted avg       1.00      1.00      1.00     45149

